# MINDTRACE — MentalBERT 7-Class Text Model Training (Google Colab / GPU)

This notebook faithfully trains and evaluates the fine-tuned MentalBERT model on the Kaggle Sentiment Analysis for Mental Health dataset (52,680 records).

### Target Classes (Exact alphabetical order):
1. **Anxiety**
2. **Bipolar**
3. **Depression**
4. **Normal**
5. **Personality disorder**
6. **Stress**
7. **Suicidal**

**Runtime Requirements:** Select **Runtime > Change runtime type > T4 GPU** (or A100/V100).

In [ ]:
# 1. Install dependencies
!pip install -q transformers torch scikit-learn pandas numpy datasets seaborn matplotlib

In [ ]:
# 2. Check GPU environment
import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)
if torch.cuda.is_available():
    print("GPU Model:", torch.cuda.get_device_name(0))
    print("VRAM (GB):", round(torch.cuda.get_device_properties(0).total_memory / (1024**3), 2))
else:
    print("WARNING: GPU not detected! Switch to GPU runtime for ~40x faster training.")

In [ ]:
# 3. Authenticate with HuggingFace (MentalBERT requires gated model access)
import os
from huggingface_hub import login

hf_token = os.getenv("HF_TOKEN")
if not hf_token:
    from getpass import getpass
    hf_token = getpass("Enter your Hugging Face Token (Read access): ")

login(token=hf_token)
print("Logged in to Hugging Face Hub successfully.")

In [ ]:
# 4. Load Sentiment Analysis for Mental Health Dataset (52,680 records)
import pandas as pd
import numpy as np
from datasets import load_dataset
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split

print("Loading dataset from Hugging Face: AhmedSSoliman/sentiment-analysis-for-mental-health-Combined-Data...")
ds = load_dataset("AhmedSSoliman/sentiment-analysis-for-mental-health-Combined-Data")["train"]
df_raw = ds.to_pandas()
print(f"Raw records: {len(df_raw)}")

# Standardize columns and drop NaN
text_col = "statement" if "statement" in df_raw.columns else "text"
label_col = "status" if "status" in df_raw.columns else "label"
df = pd.DataFrame({
    "text": df_raw[text_col],
    "label": df_raw[label_col]
}).dropna()
print(f"Clean records: {len(df)}")

# Encode target labels
le = LabelEncoder()
df["label_encoded"] = le.fit_transform(df["label"])
class_names = list(le.classes_)
print("
Classes:")
for idx, c in enumerate(class_names):
    print(f"  {idx} -> {c}")

# Clean text strings
X = [str(x).strip() if x is not None and str(x).strip() else "empty" for x in df["text"]]
y = df["label_encoded"].tolist()

# 80 / 10 / 10 stratified split (exact reproduction)
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp)
print(f"
Train: {len(X_train)} | Val: {len(X_val)} | Test: {len(X_test)}")

In [ ]:
# 5. Tokenize with MentalBERT
from transformers import AutoTokenizer
from torch.utils.data import Dataset, DataLoader

MODEL_NAME = "mental/mental-bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, token=hf_token)

class MentalHealthDataset(Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels
    def __len__(self):
        return len(self.labels)
    def __getitem__(self, idx):
        item = {key: val[idx] for key, val in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item

print("Tokenizing datasets (max_length=128)...")
def tokenize_batch(texts):
    return tokenizer(texts, padding=True, truncation=True, max_length=128, return_tensors="pt")

train_enc = tokenize_batch(X_train)
val_enc = tokenize_batch(X_val)
test_enc = tokenize_batch(X_test)

train_loader = DataLoader(MentalHealthDataset(train_enc, y_train), batch_size=16, shuffle=True)
val_loader = DataLoader(MentalHealthDataset(val_enc, y_val), batch_size=16, shuffle=False)
test_loader = DataLoader(MentalHealthDataset(test_enc, y_test), batch_size=16, shuffle=False)
print("DataLoaders ready.")

In [ ]:
# 6. Train MentalBERT (3 Epochs, AdamW lr=2e-5)
import time
from transformers import AutoModelForSequenceClassification
from torch.optim import AdamW

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(class_names),
    token=hf_token
).to(device)

optimizer = AdamW(model.parameters(), lr=2e-5)
EPOCHS = 3

print(f"Starting training for {EPOCHS} epochs on {device}...")
for epoch in range(EPOCHS):
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    start_time = time.time()
    for batch_idx, batch in enumerate(train_loader):
        optimizer.zero_grad()
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)
        outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        preds = outputs.logits.argmax(dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)
        if (batch_idx + 1) % 200 == 0:
            elapsed = time.time() - start_time
            eta = (elapsed / (batch_idx + 1)) * (len(train_loader) - batch_idx - 1)
            print(f"  Epoch {epoch+1} | Batch {batch_idx+1}/{len(train_loader)} | Loss: {loss.item():.4f} | Acc: {correct/total:.4f} | ETA: {eta/60:.1f}m")
    
    # Validation
    model.eval()
    val_correct, val_total = 0, 0
    with torch.no_grad():
        for batch in val_loader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)
            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            val_correct += (outputs.logits.argmax(dim=1) == labels).sum().item()
            val_total += labels.size(0)
    print(f"Epoch {epoch+1} Done | Train Acc: {correct/total:.4f} | Val Acc: {val_correct/val_total:.4f} | Time: {(time.time()-start_time)/60:.1f}m")

# Save final checkpoint
torch.save(model.state_dict(), "text_model_final.pt")
print("Saved state dict to text_model_final.pt")

In [ ]:
# 7. Evaluate on Hold-out Test Set
from sklearn.metrics import classification_report, accuracy_score, f1_score, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

model.eval()
all_preds, all_labels = [], []
with torch.no_grad():
    for batch in test_loader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        all_preds.extend(outputs.logits.argmax(dim=1).cpu().tolist())
        all_labels.extend(batch["labels"].tolist())

print("="*55)
print("FINAL TEST EVALUATION RESULTS")
print("="*55)
print(f"Test Accuracy : {accuracy_score(all_labels, all_preds):.4f}")
print(f"Macro F1      : {f1_score(all_labels, all_preds, average='macro'):.4f}")
print(f"Weighted F1   : {f1_score(all_labels, all_preds, average='weighted'):.4f}
")
print(classification_report(all_labels, all_preds, target_names=class_names))

# Plot Confusion Matrix
cm = confusion_matrix(all_labels, all_preds)
plt.figure(figsize=(9, 7))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=class_names, yticklabels=class_names)
plt.title("MentalBERT Test Set Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("True")
plt.tight_layout()
plt.savefig("confusion_matrix_text.png", dpi=300)
plt.show()

In [ ]:
# 8. Download Checkpoint for MINDTRACE
try:
    from google.colab import files
    print("Downloading text_model_final.pt to your local machine...")
    files.download("text_model_final.pt")
    print("Place this file in: multimodal-mental-health/models/text/text_model_final.pt")
except Exception as e:
    print(f"Saved to local directory as text_model_final.pt ({e})")